# **Building Production-Grade Data Agents on GCP**

## Notebook 1 — Build Your First Enterprise Data Agent

Build a read-only Gemini + ADK inventory-risk agent over BigQuery.

**Colab-only workflow:** open this notebook from the workshop GitHub repository and run the cells from the top. The repository is cloned and dependencies are installed automatically. You only edit the clearly marked **Participant configuration** cell; the shared setup module handles authentication, Vertex AI configuration and BigQuery preparation.

No participant terminal, local IDE, separate setup notebook or Gemini API key is required. Authentication uses your Google account through Colab ADC and Vertex AI.


In [ ]:
# Bootstrap: clone the workshop repository and install its dependencies.
# ORGANISER: replace the repository URL once before publishing the workshop.
from pathlib import Path
import subprocess

REPOSITORY_URL = "https://github.com/REPLACE_WITH_GITHUB_OWNER/enterprise-data-agents-gdg-essex.git"
REPOSITORY_REF = "main"
REPO_DIR = Path("/content/enterprise-data-agents-gdg-essex")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPOSITORY_REF,
         REPOSITORY_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print("Workshop repository already cloned in this runtime.")

%cd /content/enterprise-data-agents-gdg-essex
%pip install -q -r requirements-agent.txt
print("Repository ready. Next, edit the Participant configuration cell.")

In [ ]:
# PARTICIPANT CONFIGURATION — edit the values below, then run this cell.
# Your organiser will provide the source project and source dataset identifiers.
# Do not add passwords, API keys or other credentials here.

PARTICIPANT_CONFIG = {
    "google_cloud": {
        "project_id": "REPLACE_WITH_YOUR_GCP_PROJECT_ID",
        "bigquery_location": "US",
        "dataset_id": "essex_workshop",
    },
    "vertex_ai": {
        "model_id": "gemini-3.8-flash",
        "location": "global",
    },
    "source_data": {
        "project_id": "REPLACE_WITH_WORKSHOP_SOURCE_PROJECT_ID",
        "dataset_id": "REPLACE_WITH_WORKSHOP_SOURCE_DATASET_ID",
    },
    "repository": {
        "url": REPOSITORY_URL,
        "branch_or_tag": REPOSITORY_REF,
    },
}


### What this configuration controls

The configuration separates three things that are easy to confuse:

- **Your project** is where the workshop dataset and views are prepared.
- **BigQuery location** is the data processing location (`US` for this workshop dataset).
- **Vertex AI location** is where the model request is sent (`global` in this example).

The source project and dataset are supplied by the organiser. The setup module prepares the workshop dataset in your project. This cell writes a runtime-local YAML file; it does not change the public GitHub repository.

In [ ]:
# Review the non-secret settings that will be used by this runtime.
# This is a quick configuration sanity check before cloud resources are prepared.
for section, values in PARTICIPANT_CONFIG.items():
    print(f"\n{section}")
    for key, value in values.items():
        print(f"  {key}: {value}")

required_values = [
    PARTICIPANT_CONFIG["google_cloud"]["project_id"],
    PARTICIPANT_CONFIG["source_data"]["project_id"],
    PARTICIPANT_CONFIG["source_data"]["dataset_id"],
]
if any("REPLACE_WITH" in str(value) for value in required_values):
    raise ValueError("Replace the organiser-provided project and dataset placeholders before continuing.")
print("\nConfiguration placeholders check passed.")

# Persist the configuration in this temporary Colab clone and initialise cloud access.
import yaml
config_path = REPO_DIR / "config" / "workshop_config.yaml"
config_path.write_text(
    yaml.safe_dump(PARTICIPANT_CONFIG, sort_keys=False),
    encoding="utf-8",
)

from setup.workshop_setup import initialise_workshop
workshop = initialise_workshop(config_path=config_path, repo_dir=REPO_DIR)

bq = workshop.bigquery
PROJECT_ID = workshop.project_id
DATASET_ID = workshop.dataset_id
BIGQUERY_LOCATION = workshop.location
MODEL_ID = workshop.model_id
MODEL_LOCATION = workshop.model_location

print("Workshop setup complete.")
print(f"Project: {PROJECT_ID} | Dataset: {DATASET_ID} | BigQuery: {BIGQUERY_LOCATION}")
print(f"Gemini: {MODEL_ID} | Vertex AI location: {MODEL_LOCATION}")


## First look at the prepared BigQuery workspace

Before building an agent, inspect the data objects that the setup module made available. This is a simple operational check: it confirms that the notebook is pointed at the expected project and dataset, and makes the boundary between source tables and business-facing views visible.

In [ ]:
from google.cloud import bigquery

print(f"Project: {PROJECT_ID}")
print(f"Dataset: {DATASET_ID}")
print(f"BigQuery location: {BIGQUERY_LOCATION}")

objects = list(bq.list_tables(f"{PROJECT_ID}.{DATASET_ID}"))
for obj in sorted(objects, key=lambda item: item.table_id):
    kind = "VIEW" if obj.table_type == "VIEW" else obj.table_type
    print(f"{kind:8} {obj.table_id}")

expected_views = {
    "v_inventory_product_risk",
    "v_inventory_risk_summary",
}
available_names = {obj.table_id for obj in objects}
missing_views = expected_views - available_names
if missing_views:
    raise RuntimeError(f"Expected semantic views are missing: {sorted(missing_views)}")
print("\nRequired inventory views are present.")

## Learning objectives

By the end, you should be able to:

- Explain the difference between a chatbot response and an agent that can use tools.
- Describe the basic agent runtime (model, instructions, tools, session and runner).
- Explain **Harness Engineering**: the runtime and operational controls that make agent execution reliable.
- Use a small, allow-listed BigQuery tool surface rather than letting a model write arbitrary SQL.
- Ground an inventory-risk answer in the workshop's business definitions and query evidence.

### The scenario

You are supporting the **Essex Central Distribution Centre** (`WH-ESSEX-01`). The business wants to know which products have the greatest inventory risk and what the cost value of the safety-stock gap is.

The dataset is synthetic and uses GBP. Its snapshot date is `2026-10-01`. The first question is deliberately read-only:

> Which products are most at risk of falling below safety stock, and what is the inventory value at risk?

This notebook does not create purchase orders, change stock, approve spend or contact suppliers.

## 1. The mental model: from question to evidence-backed answer

A useful first agent is not an autonomous system with unrestricted access. It is a model that can reason over a bounded set of permitted tools and return an answer grounded in their results.

<pre>
User question
     |
     v
ADK Runner + Session  <--- Harness Engineering
     |
     v
Gemini model + instructions
     |
     +---- decides a tool is needed
     |             |
     |             v
     |      Read-only BigQuery function
     |             |
     |             v
     |      Governed semantic view
     |             |
     |             v
     |      Structured tool result
     |             |
     +<------------+
     |
     v
Evidence-grounded response
</pre>

**The key pieces**

- **Model:** interprets the request and produces a response or tool call.
- **Instructions/context:** define the role, business language, constraints and answer expectations.
- **Tool:** a narrowly scoped Python function that performs one permitted action.
- **Session:** holds the conversation state for a user interaction.
- **Runner:** executes the agent and coordinates model/tool events.
- **Harness Engineering:** the surrounding runtime, lifecycle, validation, error handling, bounded execution and tracing. It is not the same thing as the Data Harness introduced in Notebook 2.

### Define the tool contract before connecting the model

An agent tool is an interface, not just a Python function. A good contract makes its purpose and limits clear:

- one business purpose per tool;
- predictable structured output;
- no user-supplied SQL;
- read-only access to approved views;
- bounded result sizes.

We will first define and test these functions in ordinary Python. Only after that will we register them with ADK.

In [ ]:
from tools.bigquery_tools import SupplyChainBigQueryTools

supply_chain_tools = SupplyChainBigQueryTools(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    location=BIGQUERY_LOCATION,
)

def get_inventory_risk():
    """Return highest-value inventory safety-stock gaps from the latest snapshot."""
    result = supply_chain_tools.inventory_risk(
        inventory_status=None, category=None, limit=100
    )
    risk_rows = [
        row for row in result["rows"]
        if int(row.get("shortage_units") or 0) > 0
    ]
    return {
        "rows": risk_rows[:20],
        "row_count": len(risk_rows[:20]),
        "total_risk_rows_returned": len(risk_rows),
        "read_only": True,
        "note": "Rows are ordered by inventory value at risk, then shortage units.",
    }

def get_inventory_summary():
    """Return canonical inventory portfolio metrics for the latest snapshot."""
    return supply_chain_tools.inventory_summary()

print("Read-only BigQuery tools ready.")


## 2. Inspect the tool result before giving it to an agent

A good practice is to test a tool independently. This separates data-access errors from agent reasoning errors and makes the evidence visible.

In [ ]:
inventory_evidence = get_inventory_risk()
inventory_summary_evidence = get_inventory_summary()

print("Risk rows returned:", inventory_evidence["row_count"])
print("Read-only:", inventory_evidence["read_only"])
print("\nHighest-risk products:")
for row in inventory_evidence["rows"][:8]:
    print(
        f"{row['product_id']} | {row['product_name']} | "
        f"shortage={row['shortage_units']} units | "
        f"value at risk=£{float(row['inventory_value_at_risk_gbp']):,.2f}"
    )

print("\nCanonical portfolio summary:")
for row in inventory_summary_evidence["rows"]:
    print(row)

### Validate the returned evidence

This is a small data-quality gate between BigQuery and the language model. It checks the shape and basic invariants of the tool response. The checks do not prove every business rule, but they catch missing fields and obviously invalid rows before the evidence reaches the agent.

In [ ]:
required_risk_fields = {
    "product_id", "product_name", "shortage_units", "inventory_value_at_risk_gbp"
}

for index, row in enumerate(inventory_evidence["rows"]):
    missing = required_risk_fields - set(row)
    if missing:
        raise ValueError(f"Risk row {index} is missing fields: {sorted(missing)}")
    if int(row["shortage_units"]) <= 0:
        raise ValueError(f"Risk row {index} should have a positive shortage.")
    if float(row["inventory_value_at_risk_gbp"]) < 0:
        raise ValueError(f"Risk row {index} has a negative risk value.")

print(f"Validated {len(inventory_evidence['rows'])} product-risk rows.")
print("Evidence contract: required fields present; shortages positive; values non-negative.")

## Data Agent capability: translate a business question into a governed data plan

A general-purpose agent can call a database function. A **Data Agent** must first understand what the business question means: which domain, metric definitions, grain and dimensions are relevant. The semantic query planner below resolves familiar business language to approved metric IDs. It does not generate arbitrary SQL and it does not execute a query; it creates a reviewable plan for the governed Data Harness.

This is the first explicit business-question-to-evidence step in our architecture.


In [ ]:
from data_harness.core import plan_business_question

BUSINESS_QUESTION = "Which products have the largest stock shortages and inventory value at risk?"
question_plan = plan_business_question(BUSINESS_QUESTION)
print(json.dumps(question_plan, indent=2))
assert "inventory.shortage_units" in question_plan["metric_ids"]
assert question_plan["execution"] == "not_executed"


### Inspect the metric contract before trusting the result

A metric is more than a label. The catalogue records its formula, unit, entity grain, interpretation and caveats. These details help prevent common analytical errors such as summing across multiple snapshots, confusing exposure with lost revenue, or reporting a rate without its denominator.


In [ ]:
from data_harness.core import SemanticCatalogue

catalogue = SemanticCatalogue("semantic_catalogue/metrics.json")
for metric_id in question_plan["metric_ids"]:
    metric = catalogue.get_metric(metric_id)
    print(f"\n{metric['name']} [{metric['id']}]")
    print("  Formula:", metric.get("formula"))
    print("  Grain:", metric.get("entity_grain"))
    print("  Unit:", metric.get("unit"))
    print("  Caveats:", "; ".join(metric.get("caveats", [])))


### Metric selection: similar words can mean different business measures

A Data Agent must resolve the intended business concept before retrieving data. Compare shortage units with inventory value at risk: one is a physical quantity and the other is a financial exposure measure. They answer different questions and must not be substituted for one another.


In [ ]:
METRIC_COMPARISON = (
    "inventory.shortage_units",
    "inventory.value_at_risk_gbp",
)

for metric_id in METRIC_COMPARISON:
    try:
        metric = catalogue.get_metric(metric_id)
    except KeyError:
        print(f"{metric_id}: not present in this catalogue version")
        continue
    print(f"\n{metric['id']} — {metric['name']}")
    print("Meaning:", metric.get("interpretation"))
    print("Formula:", metric.get("formula"))
    print("Unit / grain:", metric.get("unit"), "/", metric.get("entity_grain"))

print("\nDiscussion: Which metric answers 'How many units are we short?' Which answers 'What is the GBP value at risk?' Why must the agent preserve the metric's unit and meaning?")


## Discover the approved schema before querying

A Data Agent should not guess column names or assume that every table is available. The Harness discovers schemas only for an explicit allow-list of business-facing views. BigQuery IAM remains the underlying security boundary; this discovery layer narrows what the workshop agent is invited to inspect.

In [ ]:
from data_harness.governance import BigQuerySchemaDiscovery

APPROVED_VIEWS = (
    'v_inventory_product_risk', 'v_inventory_risk_summary',
    'v_supplier_delivery_performance', 'v_fulfillment_order_exposure',
    'v_fulfillment_risk_summary', 'v_supplier_cross_domain_risk',
)
schema_discovery = BigQuerySchemaDiscovery(bq, PROJECT_ID, DATASET_ID, APPROVED_VIEWS)
inventory_schema = schema_discovery.discover('v_inventory_product_risk')
print(json.dumps(inventory_schema.as_dict(), indent=2))
assert inventory_schema.table_type == 'VIEW'


## Validate a query before it reaches BigQuery

The workshop tools normally use fixed, parameterised SQL. This exercise shows the next production control: a proposed query is checked for one SELECT statement, approved semantic views and a numeric row limit. It then undergoes a BigQuery dry run with a bytes-billed ceiling before execution. This is a deliberately narrow teaching validator, not a full SQL parser; IAM and BigQuery permissions still apply.

In [ ]:
from data_harness.governance import ReadOnlyQueryValidator, run_validated_bigquery_query

query_validator = ReadOnlyQueryValidator(PROJECT_ID, DATASET_ID, APPROVED_VIEWS, require_limit=True)
proposed_sql = f'''
SELECT product_id, product_name, shortage_units, inventory_value_at_risk_gbp
FROM `{PROJECT_ID}.{DATASET_ID}.v_inventory_product_risk`
WHERE shortage_units > 0
ORDER BY inventory_value_at_risk_gbp DESC
LIMIT 10
'''
validation = query_validator.validate(proposed_sql)
print(validation)
assert validation.approved


In [ ]:
query_result = run_validated_bigquery_query(
    client=bq, validator=query_validator, sql=proposed_sql,
    location=BIGQUERY_LOCATION, maximum_bytes_billed=50_000_000,
)
print('Rows returned:', query_result['row_count'])
print('Query guard:', json.dumps(query_result['validation'], indent=2))
for row in query_result['rows'][:5]:
    print(row)


### Try an unsafe query (it must be rejected)

The following examples should fail before a query is sent to BigQuery. This is the key distinction between schema discovery and authorisation: discovery tells the agent what exists; policy validation decides what it may request.

In [ ]:
unsafe_examples = [
    f"DELETE FROM `{PROJECT_ID}.{DATASET_ID}.v_inventory_product_risk` WHERE TRUE",
    f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.suppliers` LIMIT 10",
    f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.v_inventory_product_risk`",
]
for unsafe_sql in unsafe_examples:
    rejected = query_validator.validate(unsafe_sql)
    print('Approved:', rejected.approved, '| Reason:', rejected.reason)
    assert not rejected.approved


### Try a question the catalogue cannot safely resolve

A Data Agent should ask for clarification instead of guessing. This is an important difference from a generic agent that may produce a plausible-sounding answer without enough business context.


In [ ]:
AMBIGUOUS_QUESTION = "Tell me something interesting about the business."
ambiguous_plan = plan_business_question(AMBIGUOUS_QUESTION)
print(json.dumps(ambiguous_plan, indent=2))
assert ambiguous_plan["requires_clarification"] is True


## 3. Context engineering: give the agent the right business frame

The model does not automatically know what your organisation means by *available quantity*, *shortage* or *value at risk*. These definitions belong in the context supplied to the agent.

In this first notebook, context is deliberately small and explicit:

1. **Role:** Inventory Risk Analyst for one named warehouse.
2. **Business definitions:** formulas and terminology used by the business.
3. **Evidence policy:** use the supplied tools; do not invent values or causes.
4. **Action boundary:** read-only; no stock changes or purchase-order creation.
5. **Answer shape:** explain findings and retain units, dates and GBP currency.

This is context engineering in a practical form: select the information needed for the task, make it available at the right point, and keep irrelevant or unsupported assumptions out.

### Inspect the tool descriptions

ADK uses the Python function names, docstrings and signatures as part of the tool interface presented to the model. Read them as if you were reviewing an API contract for another engineering team.

In [ ]:
import inspect

for fn in (get_inventory_risk, get_inventory_summary):
    print(f"\nTool: {fn.__name__}")
    print("Purpose:", inspect.getdoc(fn))
    print("Inputs:", inspect.signature(fn))

## 4. Build the first ADK agent

Now we wrap the tested Python functions as ADK tools. The agent has a clear job, a small toolset and explicit boundaries.

Notice the separation:

- The **tool** retrieves data from approved semantic views.
- The **instructions** tell the agent how to interpret the returned evidence.
- The **ADK Runner** manages the execution and tool hand-off.

In [ ]:
from google.adk.agents import Agent
from google.adk.apps import App
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

INVENTORY_AGENT_INSTRUCTION = """
You are the Inventory Risk Analyst for Essex Central Distribution Centre
(warehouse WH-ESSEX-01).

Your task is to answer inventory-risk questions using only the supplied
read-only BigQuery tools.

Business definitions:
- Available quantity is on-hand quantity minus reserved quantity.
- Shortage units are max(safety stock minus available quantity, 0).
- Inventory value at risk is shortage units multiplied by unit cost in GBP.
- This is the cost value of the safety-stock gap, not lost revenue or realised loss.
- The workshop data contains one snapshot dated 2026-10-01; do not claim trends.

Operating rules:
1. Use get_inventory_risk for product-level evidence.
2. Use get_inventory_summary for portfolio-level totals.
3. Do not invent product names, quantities, values, dates or causes.
4. Keep monetary values in GBP and show units/denominators where relevant.
5. If a tool returns no evidence, say what is unavailable.
6. You are read-only. Never claim to update inventory, create orders, approve spend
   or contact suppliers.
7. Summarise the most important findings, cite product IDs/names and explain
   why the products are at risk using the returned fields.
"""

inventory_agent = Agent(
    name="inventory_risk_agent",
    model=MODEL_ID,
    description="Read-only inventory risk analyst for the Essex distribution centre.",
    instruction=INVENTORY_AGENT_INSTRUCTION,
    tools=[get_inventory_risk, get_inventory_summary],
)

inventory_app = App(
    name="gdg_essex_first_data_agent",
    root_agent=inventory_agent,
)
inventory_session_service = InMemorySessionService()
inventory_runner = Runner(
    app=inventory_app,
    session_service=inventory_session_service,
)

print("Agent created:", inventory_agent.name)

### Review the agent boundary

The agent should have access only to the two functions we deliberately registered. This is a useful design review before making the first model call.

In [ ]:
print("Agent:", inventory_agent.name)
print("Model:", MODEL_ID)
print("Registered tools:", [tool.__name__ for tool in (get_inventory_risk, get_inventory_summary)])
print("Read-only policy included:", "read-only" in INVENTORY_AGENT_INSTRUCTION.lower())
print("No arbitrary SQL tool registered: True")

## 5. Ask the agent a real business question

The runner creates a session and streams events. The final answer is shown below. If the model decides it needs evidence, ADK invokes one of the Python tools and returns the tool result to the model before it responds.

In [ ]:
async def ask_inventory_agent(question: str, user_id: str = "gdg_essex_participant"):
    session = await inventory_session_service.create_session(
        app_name="gdg_essex_first_data_agent",
        user_id=user_id,
    )
    message = genai_types.Content(
        role="user",
        parts=[genai_types.Part(text=question)],
    )

    final_parts = []
    event_log = []

    async for event in inventory_runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=message,
    ):
        event_log.append({
            "author": getattr(event, "author", None),
            "is_final": event.is_final_response(),
            "has_content": bool(getattr(event, "content", None)),
        })
        if event.is_final_response() and event.content and event.content.parts:
            final_parts.extend(
                part.text for part in event.content.parts
                if getattr(part, "text", None)
            )

    return {
        "answer": "\n".join(final_parts).strip(),
        "events": event_log,
        "session_id": session.id,
    }

QUESTION = (
    "Which products are most at risk of falling below safety stock? "
    "Give me the five highest inventory value-at-risk items, their shortage "
    "units and GBP value at risk, then summarise the overall inventory position. "
    "Explain what value at risk means in this dataset."
)

agent_result = await ask_inventory_agent(QUESTION)
print(agent_result["answer"])

### Try a second question with a different analytical emphasis

The same agent and tool boundary can support a different question without changing the implementation. This time, ask for operational prioritisation by shortage quantity rather than only ranking by monetary value.

In [ ]:
FOLLOW_UP_QUESTION = (
    "Which products have the largest shortage in units? "
    "Explain how the shortage is calculated and distinguish it from on-hand stock. "
    "Use only the available evidence and state the snapshot date."
)

follow_up_result = await ask_inventory_agent(FOLLOW_UP_QUESTION)
print(follow_up_result["answer"])

## 6. Review the execution evidence

The agent answer is only useful if we can inspect how it was produced. The event summary below is a first look at the ADK execution lifecycle. In Notebook 3, we will build richer tracing and multi-agent observability.

In [ ]:
print("ADK event summary")
for index, event in enumerate(agent_result["events"], start=1):
    print(
        f"{index:02d}. author={event['author']!r} "
        f"final={event['is_final']} content={event['has_content']}"
    )

if not agent_result["answer"]:
    raise RuntimeError(
        "No final agent response was captured. Review the preceding ADK event output and traceback."
    )

### What does the event sequence tell you?

ADK emits events as the request moves through the runtime. The event author can help you distinguish the user-facing agent from tool execution. The exact number and order of events can vary with model behaviour, so inspect the trace rather than assuming a fixed sequence.

In [ ]:
print("Event authors observed:")
for author in sorted({str(event["author"]) for event in agent_result["events"]}):
    print(" -", author)

print("\nEvent count:", len(agent_result["events"]))
print("Final answer captured:", bool(agent_result["answer"]))
print("Session ID:", agent_result["session_id"])

## 6. Check the answer against canonical metrics

An agent's natural-language answer should not be treated as the source of truth. Compare its portfolio figures with the deterministic BigQuery summary. The summary view is the canonical source for the workshop's inventory KPIs.

In [ ]:
from decimal import Decimal

if not inventory_summary_evidence["rows"]:
    raise RuntimeError("The inventory summary view returned no rows.")

canonical = inventory_summary_evidence["rows"][0]
print("Canonical snapshot:", canonical["snapshot_date"])
print("Warehouse:", canonical["warehouse_id"])
print("Products below safety stock:", canonical["products_below_safety_stock"])
print("Shortage units:", canonical["shortage_units"])
print("Inventory value at risk: £{:,.2f}".format(
    float(canonical["inventory_value_at_risk_gbp"])
))

print("\nInterpretation check:")
print("• Does the agent describe the figure as a safety-stock gap rather than lost revenue?")
print("• Does it use the snapshot date rather than implying a live stock position?")
print("• Can you trace each product figure to the tool result above?")

## 8. Managed Conversational Analytics — comparison, not a blocker

Google Cloud's **Conversational Analytics** is an alternative managed approach for asking natural-language questions over governed analytical data. It is useful to compare the interaction model with the custom ADK agent you just built.

| Dimension | ADK agent in this notebook | Managed Conversational Analytics |
|---|---|---|
| What you build | Agent instructions, tools, runtime and session flow | A managed conversational analytics experience over configured data |
| Tool/control surface | Explicit Python functions and allow-listed BigQuery views | Managed data/semantic configuration and service capabilities |
| Custom workflow | You control the ADK agent and can extend it with specialists | Depends on the managed service's supported integration and orchestration features |
| Workshop status | Core hands-on path | Optional comparison / demo path |

**Important:** this notebook does not call or configure the managed Conversational Analytics service. Its exact API, permissions and integration behaviour must be verified for the workshop's delivery environment. The ADK path remains complete without it.

## 9. Core Exercise, Explore Further and Advanced Challenge

### Core Exercise — complete during the workshop

1. Run the question in Section 5.
2. Identify which tool(s) the agent needed.
3. Compare the portfolio metrics with the canonical summary in Section 7.
4. Check that the answer distinguishes inventory value at risk from lost revenue.

### Explore Further

Change the question to ask for:
- The highest shortage-unit products, irrespective of cost.
- A plain-English explanation of `CRITICAL`, `AT_RISK` and `HEALTHY`.
- A summary of how available quantity differs from on-hand quantity.

Use only evidence already exposed by the tools. If the tool surface does not expose a field needed to answer, explain that limitation rather than guessing.

### Advanced Challenge

Add a new read-only tool that answers a narrowly defined inventory question. Before registering it with the agent:

- Use a fixed SQL query against an approved semantic view.
- Parameterise any filter values.
- Enforce a row limit and maximum bytes billed.
- Return structured data with a clear schema.
- Add a unit test for its query construction and boundary behaviour.

Do not add arbitrary SQL execution or write operations.

## 10. Knowledge check

1. What is the difference between a model and an agent?
2. Why is a narrow Python tool safer than allowing the model to generate arbitrary SQL?
3. Which component manages the execution and tool hand-off in this notebook?
4. What does Harness Engineering cover?
5. Why is inventory value at risk not the same as lost revenue?
6. Why should the answer be checked against the canonical BigQuery summary?

### Recap

You built a first enterprise data agent with Gemini + ADK + read-only BigQuery tools. The tool surface is constrained, business definitions are explicit, and the answer can be checked against canonical metrics.

**Next:** Notebook 2 adds a governed data-access layer, semantic catalogue, reusable Agent Skills and domain specialists.

## Where this notebook fits in the 10 workshop concepts

The workshop builds these ideas progressively. This first notebook introduces the foundations; the later notebooks extend them into reusable expertise and coordinated specialist agents.

| # | Concept | Where you encountered it | Further development |
|---:|---|---|---|
| 1 | **Agent mental model** | Model, instructions, tools, session, runner and evidence-backed response | Extended in Notebooks 2 and 3 |
| 2 | **Harness Engineering** | ADK runtime, session lifecycle, event handling and bounded execution | More robust runtime controls and observability in Notebook 3 |
| 3 | **Context Engineering** | Role, business definitions, evidence policy and action boundaries in the agent instructions | Context assembly and reusable domain context in Notebook 2 |
| 4 | **Data Harness** | Previewed through prepared BigQuery objects and approved semantic views | Governed data-access pattern and catalogue in Notebook 2 |
| 5 | **Semantic grounding** | Business definitions, canonical metrics and semantic inventory views | Shared glossary and semantic catalogue in Notebook 2 |
| 6 | **Tool engineering** | Narrow Python functions, docstrings, structured outputs and read-only boundaries | More domain tools and validation in Notebook 2 |
| 7 | **Agent Skills** | Introduced as the next step; not implemented in this notebook | Reusable domain knowledge and analytical procedures in Notebook 2 |
| 8 | **Routing and specialisation** | Not implemented yet; this notebook uses one inventory agent | Specialist selection and boundaries in Notebook 3 |
| 9 | **Orchestration and autonomy** | Not implemented yet; one agent handles the request | Delegation, coordination and human oversight in Notebook 3 |
| 10 | **Evaluation and observability** | Evidence checks, canonical-metric comparison and ADK event inspection | Broader evaluations and trace analysis in Notebook 3 |

### Concepts to take away

- A model becomes useful as an enterprise agent when it has a clear role, bounded tools and a reliable runtime.
- Business definitions and governed data views are part of the agent's context, not optional documentation.
- Tool outputs and agent prose should be checked against deterministic evidence.
- A read-only first step is a deliberate safety boundary. More autonomy should be added only with explicit controls and evaluation.